In [16]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import sys
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import ElasticNet
from sksurv.metrics import concordance_index_censored
from sksurv.linear_model import CoxnetSurvivalAnalysis
import matplotlib.pyplot as plt


sys.path.append('../utilities')

PARTICIPANT_DATA_PATH = '../participant_data/'

In [17]:
from preprocess import preprocess

index_event = "Borrow"
outcome_event = "Liquidated"
dataset_path = os.path.join(index_event, outcome_event)

train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))

X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)

In [20]:
# prepare data
X = lifelines_train_df.drop(['timeDiff', 'status'], axis=1)
y = np.array([(bool(event), time) for event, time in 
              zip(lifelines_train_df['status'], lifelines_train_df['timeDiff'])], 
              dtype=[('status', '?'), ('timeDiff', '<f8')])

# handle categorical variables
# X = pd.get_dummies(X, drop_first=True)

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# # scale features
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# CoxNet Elastic Net
cox_elastic_net = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01)
cox_elastic_net.fit(X_train, y_train)

CoxnetSurvivalAnalysis(alpha_min_ratio=0.01, l1_ratio=0.9)

In [23]:
X_train

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,sinDayOfWeek,userBorrowAvgAmountUSD,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount
431944,-0.398555,0.553651,0.149615,-0.008581,-0.604167,0.306038,0.794494,-0.156334,0.451127,-0.401863,...,-0.674545,-0.468829,-0.133015,-0.804171,-0.398468,0.177531,-0.219429,-0.523392,1.542022,0.267589
243893,-0.398558,-1.512807,-1.699127,0.833597,-0.437234,-0.186621,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.674545,-0.485252,-0.133089,-0.840991,-0.398471,-1.428875,0.577073,-0.943668,0.008258,-0.454852
354704,-0.394850,-0.195279,-0.911062,-1.073358,0.567163,0.198867,-0.793705,-0.156334,0.271913,-0.392084,...,1.050891,0.110539,-0.112161,-0.576312,-0.394824,0.543421,2.440103,-0.752688,0.008258,-0.416909
642756,0.213858,0.338450,0.720297,0.289315,-0.347154,-0.814355,1.191543,-0.156334,1.526411,0.381103,...,-0.674545,0.849960,-0.132945,-0.007848,0.213785,0.862208,-0.519926,-0.321613,0.008258,2.869641
740657,-0.397916,0.117362,1.545696,1.056196,4.009286,0.048240,0.794494,10.519896,1.347197,-0.378670,...,1.050891,-0.482871,-0.130171,1.029331,-0.398111,-1.023274,0.079067,0.509132,1.542022,-0.535975
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259178,-0.398558,-1.106723,-1.483241,1.232259,-0.797463,0.659516,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.674545,-0.485252,-0.132595,-1.016799,-0.398471,-1.427184,0.558068,-1.263986,0.008258,-0.395567
365838,5.161215,-1.564782,-1.776876,1.416369,1.267244,1.094291,1.191543,-0.156334,0.719948,5.251249,...,-1.168379,1.616713,-0.129029,-0.805940,5.161573,1.287419,0.550863,-0.760335,1.542022,-0.277782
131932,-0.398554,0.549735,-0.096552,-1.433531,-0.102037,-0.493187,-0.793705,-0.156334,-0.892979,-0.402879,...,1.050891,-0.449856,0.025704,2.077382,-0.398471,0.487373,-0.357128,1.754571,-1.525506,-0.583189
671155,-0.398556,0.633918,1.219038,1.056196,0.562126,-0.673318,-0.793705,-0.156334,-0.176123,-0.402772,...,-0.674545,-0.484444,-0.088137,1.409120,-0.398470,-0.480779,-0.498588,1.713184,1.542022,0.629833


In [ ]:
# predictions (risk scores)
risk_train = -cox_elastic_net.predict(X_train)
risk_test = cox_elastic_net.predict(X_test)

# calculate C-index
c_index_train = concordance_index_censored(y_train['status'], y_train['timeDiff'], risk_train)[0]
c_index_test = concordance_index_censored(y_test['status'], y_test['timeDiff'], risk_test)[0]

# results
print(f"Train C-index: {c_index_train:.3f}")
print(f"Test C-index: {c_index_test:.3f}")

# # get coefficients for the best alpha
# coef = cox_elastic_net.coef_
# feature_importance = pd.DataFrame({
#     'feature': X.columns,
#     'coefficient': coef
# }).sort_values('coefficient', key=abs, ascending=False)

# print(f"\nTop 10 important features:")
# print(feature_importance.head(10))

Train C-index: 0.177
Test C-index: 0.171
